# 🗄️ SQL Complete Interview Preparation — Tomorrow 9 AM
### Basic Definitions → Core Concepts → Advanced Topics → Window Functions → 50+ Interview Questions

---

> **How to use this notebook:**
> Every cell runs real SQL using Python's built-in `sqlite3` — no installation needed.
> Run the **Setup cell first**, then work through each section top to bottom.
> Every concept has a definition, explanation, working SQL code, and interview Q&A.

---

### 📚 Table of Contents
| Section | Topics |
|---------|--------|
| **0** | [Database Setup](#setup) |
| **1** | [All Basic Definitions](#definitions) |
| **2** | [DDL — CREATE, ALTER, DROP, TRUNCATE](#ddl) |
| **3** | [DML — INSERT, UPDATE, DELETE](#dml) |
| **4** | [SELECT — Querying Data](#select) |
| **5** | [WHERE — All Operators & Filters](#where) |
| **6** | [Aggregate Functions](#aggregate) |
| **7** | [GROUP BY & HAVING](#groupby) |
| **8** | [All JOIN Types](#joins) |
| **9** | [Subqueries](#subqueries) |
| **10** | [Window Functions — FULL DEEP DIVE](#window) |
| **11** | [String, Math & Date Functions](#functions) |
| **12** | [CASE WHEN](#case) |
| **13** | [CTEs — WITH clause](#cte) |
| **14** | [Indexes, Keys & Constraints](#keys) |
| **15** | [Normalization](#normalization) |
| **16** | [Transactions & ACID](#transactions) |
| **17** | [Views](#views) |
| **18** | [50+ Interview Questions & Answers](#interview) |
| **19** | [Classic Coding Problems](#coding) |
| **20** | [SQL Execution Order & Final Cheat Sheet](#cheatsheet) |

---
<a id='setup'></a>
## ⚙️ Section 0 — Database Setup
### Run this cell FIRST before anything else

In [0]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

def run(sql, title=""):
    try:
        statements = [s.strip() for s in sql.strip().split(";") if s.strip()]
        last_rows = None
        last_desc = None
        for stmt in statements:
            cur.execute(stmt)
            if cur.description:
                last_rows = cur.fetchall()
                last_desc = cur.description
        conn.commit()
        if last_rows is not None and last_desc is not None:
            if title:
                print(f"  {title}")
                print("  " + "─" * 50)
            cols = [d[0] for d in last_desc]
            widths = [max(len(str(c)), max((len(str(r[i])) for r in last_rows), default=0))
                      for i, c in enumerate(cols)]
            header = " │ ".join(str(c).ljust(widths[i]) for i, c in enumerate(cols))
            sep    = "─┼─".join("─" * w for w in widths)
            print(f"  {header}")
            print(f"  {sep}")
            for row in last_rows:
                print("  " + " │ ".join(str(row[i]).ljust(widths[i]) for i in range(len(cols))))
            print(f"  ({len(last_rows)} row{'s' if len(last_rows)!=1 else ''})")
        else:
            print(f"  ✅ Done — rows affected: {cur.rowcount}")
    except Exception as e:
        print(f"  ❌ ERROR: {e}")

cur.executescript("""
CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT    NOT NULL UNIQUE,
    city      TEXT    NOT NULL,
    budget    REAL    DEFAULT 0
);
CREATE TABLE employees (
    emp_id    INTEGER PRIMARY KEY,
    fname     TEXT    NOT NULL,
    lname     TEXT    NOT NULL,
    email     TEXT    UNIQUE,
    salary    REAL    NOT NULL,
    hire_date TEXT,
    dept_id   INTEGER,
    mgr_id    INTEGER,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
);
CREATE TABLE projects (
    proj_id   INTEGER PRIMARY KEY,
    proj_name TEXT    NOT NULL,
    dept_id   INTEGER,
    budget    REAL
);
CREATE TABLE emp_projects (
    emp_id  INTEGER,
    proj_id INTEGER,
    role    TEXT,
    PRIMARY KEY(emp_id, proj_id)
);
INSERT INTO departments VALUES (10,'Engineering','Noida',   5000000);
INSERT INTO departments VALUES (20,'Marketing',  'Delhi',   2000000);
INSERT INTO departments VALUES (30,'HR',         'Noida',   1500000);
INSERT INTO departments VALUES (40,'Finance',    'Mumbai',  3000000);
INSERT INTO departments VALUES (50,'Research',   'Bengaluru',4000000);
INSERT INTO employees VALUES (1, 'Arjun', 'Sharma','arjun@co.com', 85000,'2020-03-15',10,NULL);
INSERT INTO employees VALUES (2, 'Priya', 'Singh', 'priya@co.com', 72000,'2021-06-01',10,1);
INSERT INTO employees VALUES (3, 'Rahul', 'Gupta', 'rahul@co.com', 68000,'2021-08-20',10,1);
INSERT INTO employees VALUES (4, 'Sneha', 'Kumar', 'sneha@co.com', 91000,'2019-11-10',20,NULL);
INSERT INTO employees VALUES (5, 'Vikram','Patel', 'vikram@co.com',55000,'2022-01-05',20,4);
INSERT INTO employees VALUES (6, 'Ananya','Reddy', 'ananya@co.com',62000,'2022-03-22',30,NULL);
INSERT INTO employees VALUES (7, 'Karan', 'Mehta', 'karan@co.com', 48000,'2023-07-11',30,6);
INSERT INTO employees VALUES (8, 'Divya', 'Nair',  'divya@co.com', 95000,'2018-05-30',40,NULL);
INSERT INTO employees VALUES (9, 'Rohan', 'Joshi', 'rohan@co.com', 77000,'2020-09-14',40,8);
INSERT INTO employees VALUES (10,'Meena', 'Pillai','meena@co.com', 83000,'2021-12-01',50,NULL);
INSERT INTO employees VALUES (11,'Aditya','Shah',  'aditya@co.com',59000,'2023-02-28',10,1);
INSERT INTO employees VALUES (12,'Pooja', 'Verma', 'pooja@co.com', 71000,'2022-10-15',50,10);
INSERT INTO employees VALUES (13,'Suresh','Yadav', 'suresh@co.com',44000,'2023-11-01',30,6);
INSERT INTO employees VALUES (14,'Kavita','Mishra','kavita@co.com',66000,'2021-04-18',20,4);
INSERT INTO employees VALUES (15,'Nikhil','Tiwari',NULL,           52000,'2024-01-10',10,1);
INSERT INTO projects VALUES (101,'Customer Portal', 10,800000);
INSERT INTO projects VALUES (102,'AI Analytics',    50,1200000);
INSERT INTO projects VALUES (103,'Brand Campaign',  20,400000);
INSERT INTO projects VALUES (104,'ERP Upgrade',     40,600000);
INSERT INTO emp_projects VALUES (1, 101,'Lead');
INSERT INTO emp_projects VALUES (2, 101,'Dev');
INSERT INTO emp_projects VALUES (3, 101,'Dev');
INSERT INTO emp_projects VALUES (10,102,'Manager');
INSERT INTO emp_projects VALUES (12,102,'Analyst');
INSERT INTO emp_projects VALUES (4, 103,'Lead');
INSERT INTO emp_projects VALUES (8, 104,'Lead');
""")
conn.commit()
print("✅ Database ready! Tables: employees(15 rows), departments(5), projects(4), emp_projects(7)")
print("✅ run() helper loaded — use run('SELECT ...') throughout this notebook")

---
<a id='definitions'></a>
## 📌 Section 1 — All Basic Definitions You Must Know

---

### 🔷 What is SQL?
**SQL (Structured Query Language)** is the standard language for managing and querying relational databases.
You describe WHAT data you want — the database engine figures out HOW to get it.

---

### 🔷 What is a Database?
An organized collection of structured data stored electronically, designed for easy access, management, and updating.

---

### 🔷 What is a Relational Database?
A database that stores data in **tables** (rows & columns) where tables can be related to each other through keys.
Based on E.F. Codd's relational model (1970). Examples: MySQL, PostgreSQL, Oracle, SQL Server, SQLite.

---

### 🔷 What is a Table?
A collection of related data organized in rows and columns. Each table represents one real-world entity.
- Also called: **relation** (mathematical term)
- Rows = records = tuples
- Columns = fields = attributes

---

### 🔷 What is a Schema?
The **blueprint/structure** of a database — all tables, columns, data types, constraints, and relationships defined before inserting data.

---

### 🔷 SQL Command Categories (5 types — always asked!)

| Category | Full Name | Commands | Purpose |
|----------|-----------|----------|---------|
| **DDL** | Data Definition Language | CREATE, ALTER, DROP, TRUNCATE, RENAME | Define structure |
| **DML** | Data Manipulation Language | INSERT, UPDATE, DELETE, MERGE | Modify data |
| **DQL** | Data Query Language | SELECT | Retrieve data |
| **DCL** | Data Control Language | GRANT, REVOKE | Permissions |
| **TCL** | Transaction Control Language | COMMIT, ROLLBACK, SAVEPOINT | Transactions |

---

### 🔷 Keys (6 types — memorize all)

| Key | Definition |
|-----|-----------|
| **Primary Key** | Uniquely identifies each row. NOT NULL + UNIQUE. One per table. |
| **Foreign Key** | References PK of another table. Enforces referential integrity. |
| **Unique Key** | All values different. Allows ONE NULL. Multiple per table. |
| **Candidate Key** | Any column that COULD be a primary key (unique + not null). |
| **Composite Key** | Primary key made of two or more columns combined. |
| **Super Key** | Any set of columns that uniquely identifies a row. |

---

### 🔷 Constraints (rules enforced by the database)

| Constraint | Rule |
|------------|------|
| `PRIMARY KEY` | Unique + Not Null |
| `FOREIGN KEY` | Must match PK in referenced table |
| `NOT NULL` | Column must always have a value |
| `UNIQUE` | No duplicate values allowed |
| `DEFAULT` | Use this value if none provided |
| `CHECK` | Value must satisfy a condition |

---

### 🔷 ACID Properties (transactions — always asked!)

| Letter | Property | Meaning |
|--------|----------|---------|
| **A** | Atomicity | All operations succeed or none do — no partial updates |
| **C** | Consistency | Database moves from one valid state to another |
| **I** | Isolation | Concurrent transactions don't interfere with each other |
| **D** | Durability | Committed changes survive system crashes |

---

### 🔷 Normalization Forms

| Form | Rule |
|------|------|
| **1NF** | Atomic values only — no repeating groups, no multi-valued cells |
| **2NF** | 1NF + no partial dependency (every non-key depends on WHOLE PK) |
| **3NF** | 2NF + no transitive dependency (non-keys depend only on PK) |
| **BCNF** | Stricter 3NF — every determinant must be a candidate key |

---

### 🔷 JOIN Types

| JOIN | Returns |
|------|---------|
| INNER JOIN | Only matching rows from both tables |
| LEFT JOIN | All from left + matching from right (NULL if no match) |
| RIGHT JOIN | All from right + matching from left (NULL if no match) |
| FULL OUTER JOIN | All rows from both tables |
| SELF JOIN | Table joined with itself |
| CROSS JOIN | Every combination (cartesian product) |

---

### 🔷 Important Terms

**Index** — data structure that speeds up data retrieval (like a book index). Speeds reads, slows writes.

**View** — virtual table based on a SELECT query. No data stored. Runs query each time.

**Stored Procedure** — named, saved SQL code that can be executed with one call.

**Trigger** — SQL code that auto-executes on INSERT/UPDATE/DELETE events.

**Transaction** — group of SQL statements that execute as one unit — all or nothing.

**Subquery** — query nested inside another query. Inner runs first.

**CTE** — Common Table Expression — temporary named result set using WITH keyword.

**Window Function** — calculation across related rows without collapsing them (unlike GROUP BY).

**NULL** — represents absence of a value. Not zero, not empty string. Use IS NULL to check.

**Cardinality** — number of unique values in a column.

**Referential Integrity** — guarantee that foreign key values point to existing primary keys.

**Denormalization** — adding redundancy back for read performance (used in data warehouses).

---
<a id='ddl'></a>
## 📌 Section 2 — DDL (Data Definition Language)
**DDL defines the STRUCTURE of the database — not the data itself.**
DDL commands are auto-committed in most databases — they cannot be rolled back.

In [0]:
# ── CREATE TABLE with all constraint types ────────────────────────
run("""
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT    NOT NULL,
    category     TEXT    DEFAULT 'General',
    price        REAL    NOT NULL CHECK(price > 0),
    stock        INTEGER DEFAULT 0 CHECK(stock >= 0),
    sku          TEXT    UNIQUE
)
""", "CREATE TABLE — products with all constraints")

In [0]:
# ── ALTER TABLE — modify structure after creation ─────────────────
run("ALTER TABLE products ADD COLUMN is_active INTEGER DEFAULT 1",
    "ALTER TABLE — add column")

run("ALTER TABLE products ADD COLUMN supplier TEXT",
    "ALTER TABLE — add another column")

run("SELECT * FROM products LIMIT 1", "Verify new columns exist (empty table)")

In [0]:
# ── INSERT some data to demonstrate DROP ─────────────────────────
run("""
INSERT INTO products VALUES (1,'Laptop','Electronics',75000,10,'SKU001',1,'Dell');
INSERT INTO products VALUES (2,'Mouse', 'Electronics', 1200,50,'SKU002',1,'HP');
INSERT INTO products VALUES (3,'Desk',  'Furniture',  15000, 5,'SKU003',1,'Local')
""", "Insert sample products")

run("SELECT * FROM products", "All products")

In [0]:
# ── TRUNCATE vs DELETE vs DROP ────────────────────────────────────

# TRUNCATE — removes ALL rows, keeps structure, fast, no rollback
# DELETE    — removes specific rows, can rollback, fires triggers
# DROP      — removes entire table (structure + data) permanently

print("  Difference: TRUNCATE vs DELETE vs DROP")
print("  " + "─"*50)
print("  TRUNCATE → removes ALL rows, keeps table structure, resets auto-increment, FAST")
print("  DELETE   → removes specific rows (with WHERE), can ROLLBACK, fires triggers")
print("  DROP     → removes ENTIRE table — structure AND data — PERMANENT, cannot undo")
print()

# Demonstrate DROP
run("DROP TABLE IF EXISTS products", "DROP TABLE — removes everything permanently")

# Table is now gone
run("SELECT name FROM sqlite_master WHERE type='table'", "Remaining tables")

### 💡 Interview Questions — DDL

**Q: What is the difference between DDL and DML?**
DDL changes the **structure** (table design, columns, constraints). DML changes the **data** inside tables.

**Q: Can DDL statements be rolled back?**
In most databases (MySQL, Oracle) DDL is auto-committed and CANNOT be rolled back. PostgreSQL is an exception — DDL can be rolled back inside a transaction.

**Q: What is the difference between TRUNCATE and DELETE?**
TRUNCATE removes ALL rows instantly, resets auto-increment, cannot be rolled back, doesn't fire row-level triggers, faster.
DELETE removes specific rows with WHERE clause, can be rolled back, fires triggers, slower on large tables.

**Q: What is the difference between DROP and TRUNCATE?**
TRUNCATE keeps the table structure — just empties it. DROP removes the entire table including its structure permanently.

---
<a id='dml'></a>
## 📌 Section 3 — DML (Data Manipulation Language)
**DML works with the DATA inside tables — insert, change, remove rows.**

In [0]:
# ── INSERT ────────────────────────────────────────────────────────
run("""
INSERT INTO departments (dept_id, dept_name, city, budget)
VALUES (60, 'Legal', 'Delhi', 1000000)
""", "INSERT single row")

run("""
INSERT INTO employees (emp_id, fname, lname, email, salary, hire_date, dept_id)
VALUES
    (16, 'Test1', 'User', 'test1@co.com', 40000, '2024-01-01', 10),
    (17, 'Test2', 'User', 'test2@co.com', 42000, '2024-02-01', 20),
    (18, 'Test3', 'User', 'test3@co.com', 38000, '2024-03-01', 30)
""", "INSERT multiple rows at once")

run("SELECT emp_id, fname, salary, dept_id FROM employees WHERE emp_id >= 16",
    "Verify inserts")

In [0]:
# ── UPDATE ────────────────────────────────────────────────────────
# ALWAYS use WHERE — without it EVERY row is updated!

run("""
UPDATE employees
SET salary = 45000
WHERE emp_id = 16
""", "UPDATE single value")

run("""
UPDATE employees
SET salary    = salary * 1.10,
    hire_date = '2024-01-15'
WHERE emp_id IN (17, 18)
""", "UPDATE multiple columns + multiple rows")

run("SELECT emp_id, fname, salary, hire_date FROM employees WHERE emp_id >= 16",
    "After UPDATE")

In [0]:
# ── DELETE ────────────────────────────────────────────────────────
# ALWAYS use WHERE — without it EVERY row is deleted!

run("DELETE FROM employees WHERE emp_id >= 16", "DELETE test rows")
run("DELETE FROM departments WHERE dept_id = 60",  "DELETE test dept")

run("SELECT COUNT(*) AS remaining FROM employees", "Rows remaining")

### 💡 Interview Questions — DML

**Q: What happens if you run UPDATE without a WHERE clause?**
Every single row in the table gets updated. This is a very dangerous mistake.

**Q: What is the difference between INSERT INTO SELECT and INSERT INTO VALUES?**
`INSERT INTO VALUES` inserts hardcoded values. `INSERT INTO SELECT` copies data from another table's query result.

**Q: Can DML be rolled back?**
Yes. DML changes are not permanent until COMMITted. You can ROLLBACK to undo them within a transaction.

---
<a id='select'></a>
## 📌 Section 4 — SELECT (DQL — Data Query Language)
**SELECT is the most used SQL command. 80% of your SQL work is writing SELECT queries.**

In [0]:
# ── Basic SELECT ──────────────────────────────────────────────────
run("SELECT * FROM employees", "All columns, all rows")

In [0]:
# ── Select specific columns ───────────────────────────────────────
run("SELECT fname, lname, salary FROM employees", "Specific columns")

In [0]:
# ── Column aliases with AS ────────────────────────────────────────
run("""
SELECT
    fname                     AS "First Name",
    lname                     AS "Last Name",
    salary                    AS "Monthly (₹)",
    salary * 12               AS "Annual (₹)"
FROM employees
""", "Column aliases")

In [0]:
# ── DISTINCT — remove duplicates ──────────────────────────────────
run("SELECT dept_id FROM employees ORDER BY dept_id",
    "dept_id WITH duplicates")

run("SELECT DISTINCT dept_id FROM employees ORDER BY dept_id",
    "dept_id WITHOUT duplicates")

In [0]:
# ── ORDER BY and LIMIT ────────────────────────────────────────────
run("""
SELECT fname, salary
FROM employees
ORDER BY salary DESC
LIMIT 5
""", "Top 5 highest paid")

run("""
SELECT fname, salary
FROM employees
ORDER BY salary DESC
LIMIT 5 OFFSET 5
""", "Ranks 6-10 (OFFSET pagination)")

---
<a id='where'></a>
## 📌 Section 5 — WHERE Clause — All Operators

In [0]:
# ── Comparison operators ──────────────────────────────────────────
run("SELECT fname, salary FROM employees WHERE salary > 70000", "salary > 70000")
run("SELECT fname, salary FROM employees WHERE salary != 85000","salary != 85000")
run("SELECT fname, salary FROM employees WHERE salary BETWEEN 60000 AND 80000",
    "BETWEEN 60000 AND 80000 (inclusive)")

In [0]:
# ── IN and NOT IN ────────────────────────────────────────────────
run("SELECT fname, dept_id FROM employees WHERE dept_id IN (10, 50)",
    "dept_id IN (10, 50)")
run("SELECT fname, dept_id FROM employees WHERE dept_id NOT IN (10, 20, 50)",
    "dept_id NOT IN (10,20,50)")

In [0]:
# ── LIKE — pattern matching ───────────────────────────────────────
# % = any number of characters   _ = exactly one character
run("SELECT fname FROM employees WHERE fname LIKE 'A%'",  "Starts with A")
run("SELECT fname FROM employees WHERE fname LIKE '%a'",  "Ends with a")
run("SELECT fname FROM employees WHERE fname LIKE '%an%'","Contains 'an'")
run("SELECT fname FROM employees WHERE fname LIKE '_____'",
    "Exactly 5 characters (5 underscores)")

In [0]:
# ── NULL handling ─────────────────────────────────────────────────
run("SELECT fname, mgr_id FROM employees WHERE mgr_id IS NULL",
    "Top-level employees (no manager)")
run("SELECT fname, email FROM employees WHERE email IS NULL",
    "Employees with no email")

# NULL in arithmetic always returns NULL
run("SELECT fname, salary, salary + NULL AS result FROM employees LIMIT 3",
    "NULL in arithmetic = NULL")

In [0]:
# ── AND, OR, NOT with parentheses ────────────────────────────────
run("""
SELECT fname, salary, dept_id
FROM employees
WHERE (dept_id = 10 OR dept_id = 50)
AND salary > 70000
""", "dept 10 or 50 AND salary > 70000")

---
<a id='aggregate'></a>
## 📌 Section 6 — Aggregate Functions
**Aggregate functions compute ONE value from many rows.**

| Function | Returns |
|----------|---------|
| `COUNT(*)` | Total number of rows |
| `COUNT(col)` | Count of non-NULL values |
| `SUM(col)` | Total of numeric column |
| `AVG(col)` | Average value |
| `MAX(col)` | Highest value |
| `MIN(col)` | Lowest value |

In [0]:
# ── All aggregates together ───────────────────────────────────────
run("""
SELECT
    COUNT(*)                AS total_rows,
    COUNT(mgr_id)           AS has_manager,
    COUNT(*) - COUNT(mgr_id) AS no_manager,
    ROUND(AVG(salary), 2)   AS avg_salary,
    SUM(salary)             AS total_payroll,
    MAX(salary)             AS highest,
    MIN(salary)             AS lowest
FROM employees
""", "All aggregate functions")

In [0]:
# ── COUNT(*) vs COUNT(column) — key difference ────────────────────
run("SELECT COUNT(*) AS all_rows FROM employees",       "COUNT(*) — includes NULLs")
run("SELECT COUNT(email) AS has_email FROM employees",  "COUNT(email) — skips NULLs")
run("SELECT COUNT(mgr_id) AS has_mgr FROM employees",   "COUNT(mgr_id) — skips NULLs")

### 💡 Interview Questions — Aggregates

**Q: What is the difference between COUNT(*) and COUNT(column)?**
`COUNT(*)` counts ALL rows including those with NULLs. `COUNT(column)` counts only NON-NULL values in that column.

**Q: Can you use aggregate functions in WHERE clause?**
NO. Use HAVING instead. WHERE runs before aggregation — the aggregate result doesn't exist yet when WHERE runs.

---
<a id='groupby'></a>
## 📌 Section 7 — GROUP BY & HAVING

**GROUP BY** groups rows with the same value so you can aggregate each group.
**HAVING** filters those groups — it is WHERE for groups, runs AFTER GROUP BY.

### ⚠️ Golden Rule
- Use **WHERE** to filter **rows** (before grouping)
- Use **HAVING** to filter **groups** (after grouping)
- You CANNOT use aggregate functions in WHERE

In [0]:
# ── Basic GROUP BY ────────────────────────────────────────────────
run("""
SELECT
    dept_id,
    COUNT(*)               AS headcount,
    ROUND(AVG(salary), 2)  AS avg_salary,
    SUM(salary)            AS total_cost,
    MAX(salary)            AS top_salary
FROM employees
GROUP BY dept_id
ORDER BY avg_salary DESC
""", "Aggregates per department")

In [0]:
# ── GROUP BY with JOIN ────────────────────────────────────────────
run("""
SELECT
    d.dept_name,
    d.city,
    COUNT(e.emp_id)        AS headcount,
    ROUND(AVG(e.salary),0) AS avg_salary,
    MAX(e.salary)          AS top_salary
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name, d.city
ORDER BY avg_salary DESC
""", "Department stats with names (JOIN + GROUP BY)")

In [0]:
# ── HAVING — filter groups ────────────────────────────────────────
run("""
SELECT dept_id, COUNT(*) AS cnt
FROM employees
GROUP BY dept_id
HAVING COUNT(*) > 2
""", "Departments with MORE than 2 employees")

run("""
SELECT dept_id, ROUND(AVG(salary),0) AS avg_sal
FROM employees
GROUP BY dept_id
HAVING AVG(salary) > 65000
ORDER BY avg_sal DESC
""", "Departments where average salary > 65000")

In [0]:
# ── WHERE + GROUP BY + HAVING together ───────────────────────────
run("""
SELECT
    dept_id,
    COUNT(*)              AS headcount,
    ROUND(AVG(salary), 0) AS avg_salary
FROM employees
WHERE hire_date >= '2020-01-01'     -- 1. filter rows first
GROUP BY dept_id                    -- 2. group remaining rows
HAVING AVG(salary) > 60000          -- 3. filter groups
ORDER BY avg_salary DESC            -- 4. sort result
""", "WHERE + GROUP BY + HAVING combined")

### 💡 Interview Questions — GROUP BY & HAVING

**Q: What is the difference between WHERE and HAVING?**
WHERE filters individual rows BEFORE grouping. HAVING filters groups AFTER grouping.
Aggregate functions (SUM, AVG, COUNT) can be used in HAVING but NOT in WHERE.

**Q: Can you GROUP BY without an aggregate function?**
Yes — it works like DISTINCT. But it's unusual and better to use DISTINCT instead.

**Q: In what order does SQL execute GROUP BY queries?**
FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT

---
<a id='joins'></a>
## 📌 Section 8 — All JOIN Types
**JOINs combine rows from two or more tables based on a related column.**
**This is the most tested topic in SQL interviews.**

```
Table A      Table B
  1     ──►    1   } INNER JOIN = only these
  2     ──►    2   }
  3      ✗         } LEFT only  = in LEFT JOIN result with NULL right
           ✗   4   } RIGHT only = in RIGHT JOIN result with NULL left
```

In [0]:
# ── INNER JOIN — only matching rows ──────────────────────────────
run("""
SELECT
    e.fname,
    e.salary,
    d.dept_name,
    d.city
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name
""", "INNER JOIN — only employees WITH a department")

In [0]:
# ── LEFT JOIN — all from left + matching from right ───────────────
# Add an employee with no department to demonstrate
run("INSERT INTO employees VALUES(99,'NoDept','Person',NULL,30000,'2024-01-01',NULL,NULL)")

run("""
SELECT e.fname, e.salary, d.dept_name
FROM employees e
LEFT JOIN departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name
""", "LEFT JOIN — ALL employees, even those with no department (NULL)")

run("DELETE FROM employees WHERE emp_id = 99")

In [0]:
# ── Simulated RIGHT JOIN (all depts including empty ones) ─────────
# SQLite doesn't have RIGHT/FULL OUTER — simulate with LEFT JOIN reversed
run("""
SELECT e.fname, d.dept_name, d.city
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
ORDER BY d.dept_name
""", "RIGHT JOIN (simulated) — ALL departments, even with no employees")

In [0]:
# ── SELF JOIN — table joined with itself ──────────────────────────
# mgr_id in employees references emp_id in the SAME table
run("""
SELECT
    e.fname  AS employee,
    e.salary AS emp_salary,
    m.fname  AS manager,
    d.dept_name
FROM employees e
LEFT JOIN employees   m ON e.mgr_id   = m.emp_id
JOIN      departments d ON e.dept_id  = d.dept_id
ORDER BY d.dept_name, e.fname
""", "SELF JOIN — employee with their manager name")

In [0]:
# ── Multiple JOINs — three tables ────────────────────────────────
run("""
SELECT
    e.fname       AS employee,
    d.dept_name,
    p.proj_name,
    ep.role
FROM employees  e
JOIN emp_projects ep ON e.emp_id   = ep.emp_id
JOIN projects    p  ON ep.proj_id  = p.proj_id
JOIN departments d  ON e.dept_id   = d.dept_id
ORDER BY d.dept_name
""", "Three-table JOIN — employees + projects + departments")

### 💡 Interview Questions — JOINs

**Q: What is the difference between INNER JOIN and LEFT JOIN?**
INNER JOIN returns ONLY rows that have a match in both tables.
LEFT JOIN returns ALL rows from the left table plus matching rows from the right — unmatched right side shows NULL.

**Q: What is a SELF JOIN?**
Joining a table with itself. Used for hierarchical data like employee-manager relationships where both the employee and manager are in the same employees table.

**Q: What is a CROSS JOIN?**
Returns every possible combination of rows from both tables (cartesian product). Table A has 5 rows, Table B has 3 rows → result has 15 rows. Rarely used in practice.

**Q: What is the difference between JOIN and UNION?**
JOIN combines columns (horizontal) — adds more columns from another table.
UNION combines rows (vertical) — stacks results of two queries on top of each other.

---
<a id='subqueries'></a>
## 📌 Section 9 — Subqueries
**A subquery is a query inside another query. The inner query runs first.**

### Types of Subqueries
| Type | Returns | Used in |
|------|---------|---------|
| Scalar | Single value | WHERE, SELECT, HAVING |
| Row | Single row | WHERE with = |
| Table | Multiple rows | FROM, IN, EXISTS |
| Correlated | Depends on outer | WHERE (runs per row) |

In [0]:
# ── Scalar subquery — single value ───────────────────────────────
run("""
SELECT fname, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC
""", "Employees earning above company average")

# Show the average for reference
run("SELECT ROUND(AVG(salary),2) AS company_avg FROM employees",
    "Company average salary")

In [0]:
# ── Subquery with IN ──────────────────────────────────────────────
run("""
SELECT fname, dept_id
FROM employees
WHERE emp_id IN (SELECT emp_id FROM emp_projects)
""", "Employees assigned to at least one project")

run("""
SELECT fname
FROM employees
WHERE emp_id NOT IN (SELECT emp_id FROM emp_projects)
""", "Employees NOT assigned to any project")

In [0]:
# ── Subquery in FROM clause (inline view) ────────────────────────
run("""
SELECT dept_id, avg_sal,
    CASE
        WHEN avg_sal >= 80000 THEN 'High'
        WHEN avg_sal >= 65000 THEN 'Mid'
        ELSE 'Low'
    END AS pay_band
FROM (
    SELECT dept_id, ROUND(AVG(salary),0) AS avg_sal
    FROM employees
    GROUP BY dept_id
) AS dept_avgs
ORDER BY avg_sal DESC
""", "Subquery in FROM — classify departments by pay band")

In [0]:
# ── Correlated subquery — references outer query ─────────────────
# Runs ONCE PER ROW of outer query — can be slow on large tables
run("""
SELECT fname, salary, dept_id
FROM employees e
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
    WHERE dept_id = e.dept_id   -- references outer query!
)
ORDER BY dept_id, salary DESC
""", "Employees earning more than their OWN department average (correlated)")

### 💡 Interview Questions — Subqueries

**Q: What is a correlated subquery?**
A subquery that references a column from the outer query. It runs once for every row of the outer query, making it potentially slow. Can often be rewritten as a JOIN for better performance.

**Q: What is the difference between a subquery and a JOIN?**
A JOIN combines tables horizontally. A subquery uses one query's result inside another. JOINs are generally faster. Subqueries are sometimes more readable.

**Q: What is the difference between IN and EXISTS?**
`IN` collects all values from the subquery then checks membership.
`EXISTS` stops as soon as it finds ONE matching row — faster for large datasets.

---
<a id='window'></a>
## 📌 Section 10 — Window Functions (FULL DEEP DIVE)
### The most advanced and most impressive SQL topic in interviews

**Window functions perform calculations across a set of rows related to the current row — WITHOUT collapsing rows like GROUP BY does.**

### 🔑 Syntax Structure
```sql
function_name() OVER (
    PARTITION BY column   -- divide into groups (like GROUP BY but keeps rows)
    ORDER BY column       -- order within each partition
    ROWS/RANGE BETWEEN    -- define window frame (optional)
)
```

### 🔑 Key Difference from GROUP BY
| | GROUP BY | Window Function |
|--|----------|----------------|
| Rows in output | One per group | ALL rows kept |
| Purpose | Summarize | Calculate per row without losing detail |
| Can mix with non-agg columns | NO | YES |

### 🔑 All Window Functions
| Category | Functions |
|----------|-----------|
| **Ranking** | ROW_NUMBER(), RANK(), DENSE_RANK(), NTILE(n) |
| **Offset** | LAG(), LEAD(), FIRST_VALUE(), LAST_VALUE() |
| **Aggregate** | SUM(), AVG(), COUNT(), MAX(), MIN() OVER() |

In [0]:
# ── ROW_NUMBER — unique number, no ties ──────────────────────────
run("""
SELECT
    fname,
    salary,
    dept_id,
    ROW_NUMBER() OVER (ORDER BY salary DESC) AS company_rank,
    ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dept_rank
FROM employees
ORDER BY dept_id, dept_rank
""", "ROW_NUMBER — company rank AND department rank side by side")

In [0]:
# ── RANK vs DENSE_RANK — handling ties ───────────────────────────
# Add duplicate salary to demonstrate ties
run("INSERT INTO employees VALUES(98,'TieSal','Person','tie@co.com',85000,'2023-01-01',10,1)")

run("""
SELECT
    fname,
    salary,
    ROW_NUMBER()  OVER (ORDER BY salary DESC) AS row_num,
    RANK()        OVER (ORDER BY salary DESC) AS rank_val,
    DENSE_RANK()  OVER (ORDER BY salary DESC) AS dense_rank_val
FROM employees
ORDER BY salary DESC
LIMIT 6
""", "ROW_NUMBER vs RANK vs DENSE_RANK — spot the difference on ties!")

run("DELETE FROM employees WHERE emp_id = 98")
print()
print("  KEY DIFFERENCE:")
print("  ROW_NUMBER  — always unique: 1,2,3,4,5 — no ties")
print("  RANK        — ties get same rank, SKIPS next: 1,2,2,4,5")
print("  DENSE_RANK  — ties get same rank, NO skip:   1,2,2,3,4")

In [0]:
# ── NTILE — divide into N equal buckets ──────────────────────────
run("""
SELECT
    fname,
    salary,
    NTILE(4) OVER (ORDER BY salary DESC) AS quartile
FROM employees
ORDER BY quartile, salary DESC
""", "NTILE(4) — divide into salary quartiles (Q1=top 25%, Q4=bottom 25%)")

In [0]:
# ── LAG — access previous row value ──────────────────────────────
run("""
SELECT
    fname,
    hire_date,
    salary,
    LAG(salary, 1, 0) OVER (ORDER BY hire_date)   AS prev_hire_salary,
    salary - LAG(salary, 1, 0) OVER (ORDER BY hire_date) AS salary_diff
FROM employees
WHERE hire_date IS NOT NULL
ORDER BY hire_date
""", "LAG — compare each employee's salary with previous hire's salary")

In [0]:
# ── LEAD — access next row value ─────────────────────────────────
run("""
SELECT
    fname,
    hire_date,
    salary,
    LEAD(salary, 1, 0) OVER (ORDER BY salary DESC) AS next_lower_salary,
    salary - LEAD(salary, 1, 0) OVER (ORDER BY salary DESC) AS gap_to_next
FROM employees
ORDER BY salary DESC
""", "LEAD — gap between each employee's salary and the next lower salary")

In [0]:
# ── Running total (cumulative SUM) ───────────────────────────────
run("""
SELECT
    fname,
    hire_date,
    salary,
    SUM(salary) OVER (ORDER BY hire_date) AS cumulative_payroll
FROM employees
WHERE hire_date IS NOT NULL
ORDER BY hire_date
""", "Running total — cumulative payroll as each employee was hired")

In [0]:
# ── Moving average ────────────────────────────────────────────────
run("""
SELECT
    fname,
    salary,
    ROUND(AVG(salary) OVER (
        ORDER BY salary
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 0) AS moving_avg_3
FROM employees
ORDER BY salary
""", "Moving average — average of current row + 2 rows before it")

In [0]:
# ── Window aggregate — compare to group without collapsing ────────
run("""
SELECT
    fname,
    dept_id,
    salary,
    ROUND(AVG(salary) OVER (PARTITION BY dept_id), 0)  AS dept_avg,
    salary - ROUND(AVG(salary) OVER (PARTITION BY dept_id), 0) AS vs_dept_avg,
    ROUND(AVG(salary) OVER (), 0)                      AS company_avg,
    salary - ROUND(AVG(salary) OVER (), 0)             AS vs_company_avg
FROM employees
ORDER BY dept_id, salary DESC
""", "Every employee vs their dept avg AND company avg — all in one query")

In [0]:
# ── PARTITION BY — ranking within each department ────────────────
run("""
SELECT
    fname,
    dept_id,
    salary,
    RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rank_in_dept,
    COUNT(*) OVER (PARTITION BY dept_id)                    AS dept_size,
    SUM(salary) OVER (PARTITION BY dept_id)                 AS dept_total_salary
FROM employees
ORDER BY dept_id, rank_in_dept
""", "Full department view — rank + size + total salary per dept")

In [0]:
# ── CLASSIC INTERVIEW PROBLEM: Top N per group ───────────────────
# Get top 2 highest paid employees from EACH department
run("""
SELECT fname, dept_id, salary, rank_in_dept
FROM (
    SELECT
        fname,
        dept_id,
        salary,
        RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rank_in_dept
    FROM employees
) ranked
WHERE rank_in_dept <= 2
ORDER BY dept_id, rank_in_dept
""", "Top 2 highest paid in EACH department (classic interview problem)")

### 💡 Interview Questions — Window Functions

**Q: What is the difference between GROUP BY and window functions?**
GROUP BY collapses rows into one row per group — you lose individual row detail.
Window functions keep ALL rows but add calculated columns based on a window of rows.
With window functions you can show each employee AND their department average in the same row.

**Q: What is the difference between RANK() and DENSE_RANK()?**
Both give tied rows the same rank number. RANK() skips the next number after a tie (1,2,2,4). DENSE_RANK() never skips (1,2,2,3).

**Q: What is PARTITION BY?**
Divides rows into groups (partitions) for the window function to work within. Like GROUP BY but without collapsing rows. If omitted, the entire result set is one partition.

**Q: What is LAG() used for?**
Accesses the value from a previous row. Used to calculate period-over-period changes, differences between consecutive rows, or compare current to previous values.

**Q: What does ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW mean?**
Defines the window frame — from the very first row of the partition up to and including the current row. Used for running totals.

---
<a id='functions'></a>
## 📌 Section 11 — String, Math & Date Functions

In [0]:
# ── String Functions ──────────────────────────────────────────────
run("""
SELECT
    fname,
    lname,
    UPPER(fname)                         AS upper_name,
    LOWER(lname)                         AS lower_name,
    LENGTH(fname)                        AS name_length,
    fname || ' ' || lname                AS full_name,
    SUBSTR(fname, 1, 3)                  AS first_3,
    REPLACE(lname, 'a', '@')             AS replaced,
    TRIM('  hello  ')                    AS trimmed
FROM employees
LIMIT 5
""", "String functions")

In [0]:
# ── Math Functions ────────────────────────────────────────────────
run("""
SELECT
    fname,
    salary,
    ROUND(salary / 12, 2)    AS monthly,
    ROUND(salary * 0.1, 2)   AS bonus,
    ABS(salary - 70000)      AS diff_from_70k,
    salary % 1000            AS remainder
FROM employees
LIMIT 5
""", "Math functions")

In [0]:
# ── Date Functions (SQLite style) ─────────────────────────────────
run("""
SELECT
    fname,
    hire_date,
    SUBSTR(hire_date, 1, 4)   AS hire_year,
    SUBSTR(hire_date, 6, 2)   AS hire_month,
    CAST((JULIANDAY('now') - JULIANDAY(hire_date)) / 365 AS INTEGER) AS years_at_company
FROM employees
WHERE hire_date IS NOT NULL
ORDER BY hire_date
""", "Date functions — years at company")

---
<a id='case'></a>
## 📌 Section 12 — CASE WHEN (if-else logic in SQL)
**CASE WHEN is SQL's if-else. Used to create conditional columns.**

In [0]:
# ── CASE WHEN — conditional column ───────────────────────────────
run("""
SELECT
    fname,
    salary,
    CASE
        WHEN salary >= 85000 THEN 'Senior'
        WHEN salary >= 65000 THEN 'Mid-level'
        WHEN salary >= 50000 THEN 'Junior'
        ELSE 'Entry'
    END AS level
FROM employees
ORDER BY salary DESC
""", "CASE WHEN — classify employees by salary level")

In [0]:
# ── CASE WHEN inside aggregate — PIVOT style ─────────────────────
run("""
SELECT
    ROUND(AVG(CASE WHEN dept_id = 10 THEN salary END), 0) AS Eng_avg,
    ROUND(AVG(CASE WHEN dept_id = 20 THEN salary END), 0) AS Mkt_avg,
    ROUND(AVG(CASE WHEN dept_id = 30 THEN salary END), 0) AS HR_avg,
    ROUND(AVG(CASE WHEN dept_id = 40 THEN salary END), 0) AS Fin_avg,
    ROUND(AVG(CASE WHEN dept_id = 50 THEN salary END), 0) AS Res_avg
FROM employees
""", "CASE WHEN inside aggregate — PIVOT departments into columns")

---
<a id='cte'></a>
## 📌 Section 13 — CTEs (Common Table Expressions)
**WITH clause — give a name to a temporary result set. Makes complex queries readable.**

In [0]:
# ── Simple CTE ────────────────────────────────────────────────────
run("""
WITH high_earners AS (
    SELECT emp_id, fname, salary, dept_id
    FROM employees
    WHERE salary > 70000
)
SELECT h.fname, h.salary, d.dept_name
FROM high_earners h
JOIN departments d ON h.dept_id = d.dept_id
ORDER BY h.salary DESC
""", "Simple CTE — high earners with department names")

In [0]:
# ── Multiple CTEs ─────────────────────────────────────────────────
run("""
WITH
dept_stats AS (
    SELECT dept_id,
           ROUND(AVG(salary),0) AS avg_sal,
           COUNT(*) AS headcount
    FROM employees
    GROUP BY dept_id
),
company_avg AS (
    SELECT ROUND(AVG(salary),0) AS co_avg FROM employees
)
SELECT
    d.dept_name,
    ds.avg_sal,
    ds.headcount,
    ca.co_avg,
    ds.avg_sal - ca.co_avg AS vs_company
FROM dept_stats ds
JOIN departments d  ON ds.dept_id = d.dept_id
CROSS JOIN company_avg ca
ORDER BY ds.avg_sal DESC
""", "Multiple CTEs — dept stats vs company average")

### 💡 Interview Questions — CTEs

**Q: What is a CTE and when would you use it?**
A CTE (Common Table Expression) is a temporary named result set defined with the WITH keyword at the top of a query. Use it when a subquery is complex or needs to be referenced multiple times — CTEs make the query much more readable and maintainable.

**Q: What is the difference between a CTE and a subquery?**
A CTE is defined at the top and named — it can be referenced multiple times in the main query. A subquery is inline — it can only be used once where it's written. CTEs are more readable for complex logic.

---
<a id='keys'></a>
## 📌 Section 14 — Indexes, Keys & Constraints

In [0]:
# ── Demonstrate all constraint violations ─────────────────────────

print("  1. PRIMARY KEY violation:")
run("INSERT INTO departments VALUES (10, 'Duplicate', 'City', 0)")

print("  2. NOT NULL violation:")
run("INSERT INTO departments (dept_id, city) VALUES (99, 'Delhi')")

print("  3. UNIQUE violation:")
run("INSERT INTO departments VALUES (99, 'Engineering', 'NewCity', 0)")

print("  4. CHECK violation (if supported):")
run("INSERT INTO departments VALUES (99, 'NewDept', 'City', -100)")

In [0]:
# ── Indexes ───────────────────────────────────────────────────────
run("CREATE INDEX idx_salary ON employees(salary)",
    "Create index on salary — speeds up salary-based queries")

run("CREATE INDEX idx_dept ON employees(dept_id)",
    "Create index on dept_id — speeds up JOIN on dept_id")

run("SELECT name, tbl_name FROM sqlite_master WHERE type='index'",
    "All indexes in database")

print()
print("  When to USE indexes:")
print("  ✅ Columns used frequently in WHERE clause")
print("  ✅ Columns used in JOIN conditions")
print("  ✅ Columns used in ORDER BY on large tables")
print()
print("  When NOT to use indexes:")
print("  ❌ Small tables (full scan is faster)")
print("  ❌ Columns with very few unique values (e.g. gender: M/F)")
print("  ❌ Tables with very frequent INSERT/UPDATE/DELETE (index maintenance cost)")

### 💡 Interview Questions — Keys & Indexes

**Q: Can a table have more than one primary key?**
No. A table can have only ONE primary key. But a primary key can be composite — made of multiple columns.

**Q: Can a foreign key be NULL?**
Yes. A NULL foreign key means the row simply has no relationship to the parent table.

**Q: What is the downside of indexes?**
Indexes speed up reads (SELECT) but slow down writes (INSERT, UPDATE, DELETE) because every write must also update all indexes on that table. They also take extra storage space.

**Q: What is a clustered vs non-clustered index?**
Clustered index determines the physical order of data in the table — one per table, usually the primary key.
Non-clustered index is a separate structure pointing to the data — multiple allowed per table.

---
<a id='normalization'></a>
## 📌 Section 15 — Normalization
**Organizing a database to reduce data redundancy and improve data integrity.**

In [0]:
# ── Show why normalization matters ───────────────────────────────

# BAD — unnormalized table
run("""
CREATE TABLE bad_orders (
    order_id     INTEGER,
    customer     TEXT,
    customer_city TEXT,
    products     TEXT,
    dept_name    TEXT,
    dept_budget  REAL
)
""")

run("""
INSERT INTO bad_orders VALUES
(1,'Alice','Delhi',  'Pen,Book,Bag','Sales',500000),
(2,'Bob',  'Mumbai', 'Laptop',      'IT',   800000),
(3,'Alice','Delhi',  'Pen',         'Sales',500000)
""")

run("SELECT * FROM bad_orders")

print()
print("  Problems with this table:")
print("  1NF violation : products column has 'Pen,Book,Bag' — multiple values in one cell")
print("  3NF violation : dept_budget depends on dept_name, not on order_id")
print("  Redundancy    : Alice's city stored in EVERY row — if she moves, update MANY rows")

In [0]:
# ── Our normalized schema IS already in 3NF ──────────────────────
print("  Our employees + departments schema IS normalized (3NF):")
print()
print("  employees table:")
print("  - emp_id (PK), fname, lname, salary → all depend on emp_id")
print("  - dept_id (FK) → links to departments, NOT storing dept_name here")
print()
print("  departments table:")
print("  - dept_id (PK), dept_name, city, budget → all depend on dept_id")
print()
print("  Why this is good:")
print("  ✅ If Engineering moves to Mumbai — update ONE row in departments")
print("  ✅ No dept_name repeated in every employee row")
print("  ✅ No risk of inconsistent department data")
print()
print("  1NF : All values are atomic (single value per cell) ✅")
print("  2NF : No partial dependency (no composite PK here) ✅")
print("  3NF : No transitive dependency (salary depends on emp_id, not dept) ✅")

### 💡 Interview Questions — Normalization

**Q: What is normalization and why do we do it?**
Normalization is organizing a database to eliminate data redundancy and prevent data anomalies (insertion, update, deletion anomalies). It ensures consistency and integrity.

**Q: What is a transitive dependency (3NF violation)?**
When column C depends on column B which depends on column A (the PK). So C is transitively dependent on A through B. Example: storing dept_budget in employees table — it depends on dept_id, not emp_id.

**Q: What is denormalization?**
Intentionally adding redundancy back into a database to improve read performance. Used in data warehouses and reporting databases where joins are too slow. Trade-off: faster reads, harder to maintain.

---
<a id='transactions'></a>
## 📌 Section 16 — Transactions & ACID Properties

In [0]:
# ── Transaction demo — bank transfer ─────────────────────────────
run("""
CREATE TABLE accounts (
    acc_id  INTEGER PRIMARY KEY,
    owner   TEXT,
    balance REAL CHECK(balance >= 0)
)
""")

run("INSERT INTO accounts VALUES (1, 'Arjun', 50000)")
run("INSERT INTO accounts VALUES (2, 'Priya', 30000)")

run("SELECT * FROM accounts", "Initial balances")

In [0]:
# ── Successful transaction ────────────────────────────────────────
import sqlite3 as _sq

def transfer(from_id, to_id, amount):
    try:
        cur.execute("BEGIN")
        cur.execute("UPDATE accounts SET balance = balance - ? WHERE acc_id = ?",
                    (amount, from_id))
        cur.execute("UPDATE accounts SET balance = balance + ? WHERE acc_id = ?",
                    (amount, to_id))
        conn.commit()
        print(f"  ✅ COMMIT — Transfer of ₹{amount} successful!")
    except Exception as e:
        conn.rollback()
        print(f"  ❌ ROLLBACK — Transfer failed: {e}")

print("Transferring ₹10000 from Arjun to Priya...")
transfer(1, 2, 10000)
run("SELECT * FROM accounts", "After transfer")

In [0]:
# ── Failed transaction — ROLLBACK ────────────────────────────────
print("Attempting to overdraw Arjun by ₹1,00,000...")
transfer(1, 2, 100000)   # Arjun only has 40000 — CHECK constraint fails
run("SELECT * FROM accounts", "After failed transfer — unchanged!")

In [0]:
print("""
  ACID Properties:
  ────────────────────────────────────────────────
  A — Atomicity    : Both UPDATEs succeed or NEITHER does
                     (can't have money leave one account without reaching other)

  C — Consistency  : CHECK(balance >= 0) is never violated
                     Database always stays in a valid state

  I — Isolation    : If two transfers happen simultaneously
                     they don't see each other's partial changes

  D — Durability   : Once COMMITted, the transfer is permanent
                     Even if server crashes immediately after
  ────────────────────────────────────────────────
""")

---
<a id='views'></a>
## 📌 Section 17 — Views

In [0]:
# ── Create and use a view ─────────────────────────────────────────
run("""
CREATE VIEW emp_summary AS
SELECT
    e.emp_id,
    e.fname || ' ' || e.lname AS full_name,
    e.salary,
    d.dept_name,
    d.city,
    CASE
        WHEN e.salary >= 85000 THEN 'Senior'
        WHEN e.salary >= 65000 THEN 'Mid-level'
        ELSE 'Junior'
    END AS level
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
""", "CREATE VIEW — emp_summary")

# Use like a table
run("SELECT * FROM emp_summary ORDER BY salary DESC", "Query the view")

In [0]:
# ── Filter the view ──────────────────────────────────────────────
run("SELECT full_name, salary FROM emp_summary WHERE level = 'Senior'",
    "Filter view — Senior employees only")

run("SELECT dept_name, COUNT(*) AS cnt, ROUND(AVG(salary),0) AS avg_sal FROM emp_summary GROUP BY dept_name",
    "Aggregate on view")

### 💡 Interview Questions — Views

**Q: What is a view and what are its benefits?**
A view is a virtual table based on a stored SELECT query. No data is stored.
Benefits: simplicity (hide complex JOINs), security (expose only certain columns), reusability (define logic once).

**Q: Can you INSERT/UPDATE data through a view?**
Sometimes — only if the view maps directly to one table, has no aggregation, GROUP BY, DISTINCT, or JOINs. Complex views are not updatable.

---
<a id='interview'></a>
## 📌 Section 18 — 50+ Interview Questions & Answers

---

### 🔵 Basic Level (Questions 1–20)

**Q1: What is SQL?**
Structured Query Language — standard language for managing relational databases. Used to create, read, update, and delete data.

**Q2: What are the 5 SQL command categories?**
DDL (structure), DML (data), DQL (query), DCL (permissions), TCL (transactions).

**Q3: What is a primary key?**
Column that uniquely identifies each row. Cannot be NULL or duplicate. One per table.

**Q4: What is a foreign key?**
Column that references the primary key of another table. Enforces referential integrity.

**Q5: What is referential integrity?**
Guarantee that a foreign key value either matches an existing PK value or is NULL. No orphan records.

**Q6: Difference between CHAR and VARCHAR?**
CHAR is fixed-length — always uses declared size. VARCHAR is variable-length — uses only needed space. CHAR is slightly faster, VARCHAR saves storage.

**Q7: Difference between DELETE, TRUNCATE, DROP?**
DELETE: specific rows, can rollback, fires triggers. TRUNCATE: all rows, fast, no rollback. DROP: entire table permanently.

**Q8: What is a NULL value?**
Represents absence of a value — not zero, not empty string. Use IS NULL / IS NOT NULL. NULL in arithmetic = NULL.

**Q9: Difference between WHERE and HAVING?**
WHERE filters rows before grouping. HAVING filters groups after grouping. Cannot use aggregates in WHERE.

**Q10: What is the difference between UNION and UNION ALL?**
UNION removes duplicates (slower). UNION ALL keeps all rows (faster, no deduplication step).

**Q11: What is a JOIN?**
Combines rows from two or more tables based on a related column condition.

**Q12: Difference between INNER JOIN and LEFT JOIN?**
INNER JOIN: only matching rows. LEFT JOIN: all rows from left table plus matches from right (NULL for no match).

**Q13: What is a SELF JOIN?**
A table joined with itself. Used for hierarchical data like employee-manager relationships.

**Q14: What is a CROSS JOIN?**
Returns every combination of rows from both tables (cartesian product). A×B rows = A rows × B rows.

**Q15: What is a subquery?**
A query nested inside another query. Inner query runs first. Result used by outer query.

**Q16: What is the SQL execution order?**
FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT

**Q17: Why can't you use a SELECT alias in WHERE?**
WHERE executes before SELECT in the logical order — the alias doesn't exist yet when WHERE runs.

**Q18: What is normalization?**
Organizing database to reduce redundancy and improve integrity. 1NF = atomic values, 2NF = no partial dependency, 3NF = no transitive dependency.

**Q19: What are ACID properties?**
Atomicity, Consistency, Isolation, Durability — guarantee reliable database transactions.

**Q20: What is an index?**
Data structure that speeds up data retrieval. Speeds reads, slows writes, takes extra storage.

---

### 🟡 Intermediate Level (Questions 21–35)

**Q21: What is a CTE (Common Table Expression)?**
Temporary named result set defined with WITH keyword. Makes complex queries readable. Can be referenced multiple times in one query.

**Q22: Difference between CTE and subquery?**
CTE is defined at the top and named — reusable multiple times. Subquery is inline — used once. CTEs are more readable for complex logic.

**Q23: What is a view?**
Virtual table based on a stored SELECT query. No data stored — runs query each time. Benefits: simplicity, security, reusability.

**Q24: Difference between view and table?**
Table physically stores data. View is a saved SELECT — no data stored. Querying a view executes the underlying query.

**Q25: What is a stored procedure?**
Named, precompiled set of SQL statements stored in DB. Executed with one call. Reduces repetition, improves performance.

**Q26: Difference between stored procedure and function?**
Function must return a value and can be used in SELECT. Stored procedure may or may not return values, called with EXEC/CALL.

**Q27: What is a trigger?**
SQL code that auto-executes BEFORE or AFTER INSERT/UPDATE/DELETE on a table. Used for auditing, validation, cascading changes.

**Q28: What is a composite key?**
A primary key made of two or more columns combined. Neither column is unique alone, but their combination is unique.

**Q29: What is denormalization?**
Intentionally adding redundancy to improve read performance. Used in data warehouses. Trade-off: faster reads, harder to maintain consistency.

**Q30: What is the difference between correlated and non-correlated subquery?**
Non-correlated subquery runs once — independent of outer query. Correlated subquery runs once per row — references outer query column. Correlated is slower.

**Q31: What is EXISTS vs IN?**
EXISTS stops as soon as it finds one match — faster for large subqueries.
IN collects all values from subquery then checks membership.

**Q32: What is a clustered vs non-clustered index?**
Clustered: determines physical data order, one per table. Non-clustered: separate structure pointing to data, multiple allowed.

**Q33: What are isolation levels in SQL?**
Read Uncommitted, Read Committed, Repeatable Read, Serializable — controls how transactions interact with each other's changes.

**Q34: What is a deadlock?**
Two transactions each waiting for the other to release a lock — circular wait. Database detects and kills one transaction.

**Q35: What is a surrogate key vs natural key?**
Natural key: uses real data (SSN, email). Surrogate key: artificial auto-increment ID. Surrogate keys are usually preferred — stable, no business meaning changes.

---

### 🔴 Advanced Level (Questions 36–50)

**Q36: What is the difference between RANK(), DENSE_RANK(), ROW_NUMBER()?**
ROW_NUMBER: always unique numbers, no ties (1,2,3,4,5).
RANK: tied rows get same rank, next rank skips (1,2,2,4,5).
DENSE_RANK: tied rows get same rank, no skipping (1,2,2,3,4).

**Q37: What is PARTITION BY in window functions?**
Divides rows into groups (partitions) for window function calculation. Like GROUP BY but keeps all rows. If omitted, entire result set is one partition.

**Q38: What is LAG() and when do you use it?**
Accesses value from a previous row without a self-join. Used for period-over-period comparison, detecting changes, calculating differences between consecutive rows.

**Q39: How do window functions differ from GROUP BY?**
GROUP BY collapses rows into one per group. Window functions keep ALL rows and add calculated columns — you can see individual rows AND group-level calculations together.

**Q40: What is ROWS BETWEEN in window functions?**
Defines the window frame — which rows to include in calculation.
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW = from first row to current (running total).
ROWS BETWEEN 2 PRECEDING AND CURRENT ROW = current + 2 rows before (moving average).

**Q41: How would you find the top N records per group?**
Use ROW_NUMBER() or RANK() with PARTITION BY, then wrap in subquery or CTE and filter WHERE rank <= N.

**Q42: What is query optimization?**
Rewriting queries to run faster: use indexes, avoid SELECT *, avoid functions on indexed columns in WHERE, use EXISTS over IN for large subqueries, minimize shuffles in JOINs.

**Q43: What is a covering index?**
An index that contains all columns needed by the query — query is satisfied entirely from the index without touching the actual table.

**Q44: What is EXPLAIN / EXPLAIN PLAN?**
Shows how the database engine executes a query — which indexes are used, join methods, row estimates. Used for performance tuning.

**Q45: What is the N+1 query problem?**
Fetching a list of N items, then running 1 query per item to fetch related data = N+1 queries total. Solved by using JOINs to fetch all data in one query.

**Q46: What is a materialized view?**
Like a view but data IS physically stored and periodically refreshed. Faster to query than a regular view but may be slightly stale. Not available in all databases.

**Q47: What is MERGE/UPSERT?**
Single statement that performs INSERT if row doesn't exist, UPDATE if it does. Used heavily in SCD Type 2 and data warehouse loads.

**Q48: What is temporal data and how do you handle it?**
Data that tracks changes over time. Use SCD Type 2 pattern: start_date, end_date, is_current columns to maintain full history.

**Q49: What is the difference between OLTP and OLAP?**
OLTP: Online Transaction Processing — many small fast transactions (banking, e-commerce). Normalized.
OLAP: Online Analytical Processing — complex queries on large historical data (reports, dashboards). Often denormalized.

**Q50: How would you optimize a slow SQL query?**
1. Run EXPLAIN to see execution plan. 2. Add indexes on WHERE/JOIN columns. 3. Avoid SELECT *. 4. Avoid functions on indexed columns in WHERE. 5. Replace correlated subqueries with JOINs. 6. Use EXISTS instead of IN for large sets. 7. Partition large tables. 8. Consider materialized views for complex repeated queries.

---
<a id='coding'></a>
## 📌 Section 19 — Classic SQL Coding Problems
**These are the exact problems asked in SQL interviews. Practice writing from memory.**

In [0]:
# ── Problem 1: Second highest salary ─────────────────────────────
run("""
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees)
""", "P1: Second highest salary")

In [0]:
# ── Problem 2: Nth highest salary (3rd highest) ──────────────────
run("""
SELECT DISTINCT salary AS third_highest
FROM employees
ORDER BY salary DESC
LIMIT 1 OFFSET 2
""", "P2: 3rd highest salary (OFFSET 2 = skip top 2)")

In [0]:
# ── Problem 3: Find duplicate rows ───────────────────────────────
run("INSERT INTO employees VALUES(98,'Arjun','Copy','copy@co.com',50000,'2024-01-01',10,1)")

run("""
SELECT fname, lname, COUNT(*) AS occurrences
FROM employees
GROUP BY fname, lname
HAVING COUNT(*) > 1
""", "P3: Find duplicate names")

run("DELETE FROM employees WHERE emp_id = 98")

In [0]:
# ── Problem 4: Employees earning more than their manager ──────────
run("""
SELECT
    e.fname  AS employee,
    e.salary AS emp_salary,
    m.fname  AS manager,
    m.salary AS mgr_salary
FROM employees e
JOIN employees m ON e.mgr_id = m.emp_id
WHERE e.salary > m.salary
""", "P4: Employees earning more than their manager")

In [0]:
# ── Problem 5: Department with highest average salary ─────────────
run("""
SELECT d.dept_name, ROUND(AVG(e.salary),0) AS avg_salary
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
GROUP BY d.dept_id, d.dept_name
ORDER BY avg_salary DESC
LIMIT 1
""", "P5: Department with highest average salary")

In [0]:
# ── Problem 6: Count employees per dept incl. empty depts ─────────
run("""
SELECT d.dept_name, COUNT(e.emp_id) AS employee_count
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name
ORDER BY employee_count DESC
""", "P6: Employee count per dept INCLUDING departments with 0 employees")

In [0]:
# ── Problem 7: Employees NOT on any project ──────────────────────
run("""
SELECT fname
FROM employees
WHERE emp_id NOT IN (SELECT DISTINCT emp_id FROM emp_projects)
ORDER BY fname
""", "P7: Employees not assigned to any project")

In [0]:
# ── Problem 8: Top 2 salaries per department ─────────────────────
run("""
SELECT fname, dept_id, salary, rank_in_dept
FROM (
    SELECT
        fname, dept_id, salary,
        RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rank_in_dept
    FROM employees
) ranked
WHERE rank_in_dept <= 2
ORDER BY dept_id, rank_in_dept
""", "P8: Top 2 salaries per department (window function)")

In [0]:
# ── Problem 9: Running total of payroll ──────────────────────────
run("""
SELECT
    fname,
    hire_date,
    salary,
    SUM(salary) OVER (ORDER BY hire_date) AS cumulative_payroll
FROM employees
WHERE hire_date IS NOT NULL
ORDER BY hire_date
""", "P9: Cumulative payroll as employees were hired over time")

In [0]:
# ── Problem 10: Salary percentage of total ───────────────────────
run("""
SELECT
    fname,
    salary,
    ROUND(salary * 100.0 / SUM(salary) OVER (), 2) AS pct_of_total
FROM employees
ORDER BY pct_of_total DESC
""", "P10: Each employee's salary as % of total payroll")

In [0]:
# ── Problem 11: Find employees hired in the last 2 years ──────────
run("""
SELECT fname, hire_date
FROM employees
WHERE CAST(SUBSTR(hire_date,1,4) AS INTEGER) >= 2023
ORDER BY hire_date DESC
""", "P11: Employees hired in 2023 or later")

In [0]:
# ── Problem 12: Manager with most direct reports ─────────────────
run("""
SELECT m.fname AS manager, COUNT(e.emp_id) AS direct_reports
FROM employees e
JOIN employees m ON e.mgr_id = m.emp_id
GROUP BY m.emp_id, m.fname
ORDER BY direct_reports DESC
LIMIT 1
""", "P12: Manager with most direct reports")

---
<a id='cheatsheet'></a>
## 📌 Section 20 — SQL Execution Order & Final Cheat Sheet

In [0]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           SQL LOGICAL EXECUTION ORDER                           ║
╠══════════════════════════════════════════════════════════════════╣
║  Writing order  →  SELECT FROM WHERE GROUP HAVING ORDER LIMIT   ║
║  Execution order:                                               ║
║    1. FROM          Which table(s)                              ║
║    2. JOIN          Combine tables                              ║
║    3. WHERE         Filter individual rows                      ║
║    4. GROUP BY      Group rows                                  ║
║    5. HAVING        Filter groups                               ║
║    6. SELECT        Choose columns                              ║
║    7. DISTINCT      Remove duplicates                           ║
║    8. ORDER BY      Sort results                                ║
║    9. LIMIT/OFFSET  Restrict rows                               ║
╠══════════════════════════════════════════════════════════════════╣
║  KEY FACTS TO REMEMBER                                          ║
║  ─────────────────────────────────────────────────────────────  ║
║  WHERE vs HAVING   : WHERE = rows before group                  ║
║                      HAVING = groups after group                ║
║  DELETE vs TRUNCATE: DELETE = specific rows, rollback-able      ║
║                      TRUNCATE = all rows, fast, no rollback     ║
║  RANK vs DENSE_RANK: RANK skips (1,2,2,4)                       ║
║                      DENSE_RANK no skip (1,2,2,3)               ║
║  IN vs EXISTS      : EXISTS stops at first match — faster       ║
║  JOIN vs UNION     : JOIN = columns (horizontal)                ║
║                      UNION = rows (vertical)                    ║
║  NULL rules        : NULL = NULL → UNKNOWN (not TRUE/FALSE)     ║
║                      Use IS NULL, never = NULL                  ║
╠══════════════════════════════════════════════════════════════════╣
║  5 SQL CATEGORIES  DDL DML DQL DCL TCL                          ║
║  4 ACID PROPERTIES Atomicity Consistency Isolation Durability   ║
║  3NF FORMS         1NF=atomic 2NF=no partial 3NF=no transitive  ║
║  JOIN TYPES        INNER LEFT RIGHT FULL SELF CROSS             ║
╠══════════════════════════════════════════════════════════════════╣
║         🎯 Interview Tomorrow 9 AM — You've Got This!           ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## 🌟 Final Checklist Before Tomorrow

### Tonight — what to do right now
- [ ] Run every code cell top to bottom in this notebook
- [ ] Re-read Section 1 (all definitions) — say each one out loud
- [ ] Practice writing 5 queries by hand on paper — no looking
- [ ] Know these 5 by heart: execution order, WHERE vs HAVING, RANK vs DENSE_RANK, DELETE vs TRUNCATE, JOIN types

### Tomorrow morning — what to say
When they ask a question, always answer in this order:
1. One-line definition
2. Real-world example or analogy
3. The syntax or code

### If you don't know something — say this:
*"I haven't worked with that specific case, but my approach would be to..."*
Never go silent. Always show your thinking process.

---
*Best of luck tomorrow at 9 AM! 💪🎯*